# 财务数据批量同步（手动调度）

**用途**：前端零改动，通过本 notebook 手动调取 ams_trae 后端的单只同步接口，按 symbol 列表批量更新财务 / 估值 / 市值数据。

**关键设计**
1. **限速**：每一次 HTTP 请求之后随机停顿 1–3 秒（而非每个 symbol 只停一次），避免 gm 接口锁定——同一 symbol 的多张表之间也会留出间隔。
2. **当日去重**：一个 symbol 的全部目标表都返回 success / no_data 后，调 `POST /bds/sync/mark-synced` 把 `bds_symbol_info.sync_at` 写为当日；下次运行时 `EXCLUDE_SYNCED_TODAY=True` 直接跳过当日已跑完的标的。
3. **重跑安全**：不做批次断点记录，中断后直接重跑整批即可 —— 同步接口按增量起点（`MAX(date)+1`）推进，已同步的表返回 `no_data`，幂等安全。

**前置条件**
1. ams_trae 后端已启动：`conda run -n fast311 uvicorn server_fast.main:app --host 0.0.0.0 --port 8008`
2. gm 终端已登录启动（否则同步返回 `error` / `no_data`）

**运行顺序**：配置 → 选择 symbol 来源 → 执行批量同步 → 结果汇总。失败项用末尾的 Cell 重试。


## 接口与口径约定

- **同步接口**：`POST /api/v1/bds/sync/{table}?symbol=xxx`（单只单表）。
- **标记接口**：`POST /api/v1/bds/sync/mark-synced?symbol=xxx`（写 `bds_symbol_info.sync_at = 服务器本地当日`）。
- **表名 `table`**：`fund-balance` / `fund-income` / `fund-cashflow` / `finance-deriv` / `daily-valuation` / `daily-mktvalue`。
- **返回**：HTTP 200，body `status` 三态 —— `success`（有新数据）/ `no_data`（无新数据，或 gm 未启动）/ `error`（失败）。**必须按 body.status 判断**，不能只看 HTTP 状态码。
- **增量**：财报按 `rpt_date` 增量（带 pub_date 保留策略）；每日估值 / 市值按 `trade_date` 增量。**重复运行幂等安全**。
- **首次全量**：库中无该 symbol 数据时从 `Config.EARLIEST_START_DATE`（1901-01-01）起拉取，各数据源自会截断到个股上市日 / 最早可得财报（平安银行财报可追到 1989-12-31），单只耗时明显变长。
- **当日去重**：`sync_at` 是 symbol 级标记，语义为「该标的当日已跑完全部目标表」。`no_data` 也算跑过（避免 gm 未启动时死循环重跑），`error` 不标记。
- **报告期预判**（2026-10-13 晚）：三大报表 + 财务指标在调 gm 前，先比对库内该表该标的 `MAX(rpt_date)` 与 `bds_symbol_info.latest_rpt_date`；已追平则直接返回 `no_data`（`message`=「已是最新报告期，无需导入」），**不发 gm 请求**。任一方为空则照常请求。需要强刷（财报更正回补 / 快报转正式）时给请求加 `force=true`。每日估值 / 市值按 `trade_date` 增量，不参与预判。- **重跑**：不做批次断点文件；中断后直接重跑，靠增量起点幂等跳过（已同步的表返回 `no_data`）。


In [5]:
# ===================== 配置 =====================
import time
import random

import requests
import pandas as pd

# 后端地址（ams_trae server_fast 默认 8008）
API_BASE = 'http://127.0.0.1:8008/api/v1'

# 要同步的表：4 张财报 + 每日估值 / 市值（默认全跑）
TABLES = [
    'fund-balance',     # 资产负债表
    'fund-income',      # 利润表
    'fund-cashflow',    # 现金流量表
    'finance-deriv',    # 财务指标
    'daily-valuation',  # 每日估值
    'daily-mktvalue',   # 每日市值
]

# 停顿：每次 HTTP 请求后随机停顿（秒），避免 gm 接口锁定
SLEEP_MIN, SLEEP_MAX = 1.0, 3.0

# 超时与重试
REQUEST_TIMEOUT = 60      # 单只单表超时（秒）。首次全量（1901 起点）可能超过 60s；
                          # 客户端超时不会取消服务端任务，立即重试会与服务端并发写同一 symbol
MAX_RETRY = 2             # 单只单表失败重试次数
RETRY_BACKOFF = 5.0       # 重试间隔基数（第 n 次重试前停 RETRY_BACKOFF*n 秒）

# 连续失败退避：连续失败达到阈值后额外停顿一次
FAIL_STREAK_LIMIT = 5     # 连续失败次数阈值
FAIL_STREAK_SLEEP = 15.0  # 额外停顿秒数

# 结果落盘
RESULT_CSV = 'sync_results.csv'

# symbol 前缀白名单（过滤非股票标的）
MARKET_PREFIXES = ('SHSE.', 'SZSE.', 'BJSE.')

SESSION = requests.Session()
print('配置就绪 | 后端:', API_BASE)
print('待同步表:', TABLES)


配置就绪 | 后端: http://127.0.0.1:8008/api/v1
待同步表: ['fund-balance', 'fund-income', 'fund-cashflow', 'finance-deriv', 'daily-valuation', 'daily-mktvalue']


In [6]:
# ===================== 工具函数 =====================

def _api_get(path, **params):
    '''GET 后端接口并返回 JSON。'''
    r = SESSION.get(f'{API_BASE}{path}', params=params, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()


def _fetch_paged(path, page_size=1000, **params):
    '''分页拉取列表接口，返回全部 items（自动翻页直到取完）。'''
    items, offset = [], 0
    while True:
        j = _api_get(path, limit=page_size, offset=offset, **params)
        batch = j.get('items', [])
        items.extend(batch)
        total = j.get('total', 0)
        offset += page_size
        if not batch or offset >= total:
            break
    return items


def _filter_market(symbols):
    '''仅保留白名单前缀的股票标的，去重并保持顺序。'''
    seen, out = set(), []
    for s in symbols:
        if s and s.startswith(MARKET_PREFIXES) and s not in seen:
            seen.add(s)
            out.append(s)
    return out


def resolve_symbols(scope, explicit=None, industry='', index_code='', max_symbols=0,
                    exclude_synced_today=False):
    '''按 scope 解析 symbol 列表，可选剔除「当日已同步」的标的。

    「当日」由后端 /bds/symbol-infos?exclude_synced_today=1 判定，
    依据是 bds_symbol_info.sync_at == 服务器本地当日。
    scope: explicit | all | industry | index
    '''
    scope = scope.lower()
    _only_today = 1 if exclude_synced_today else 0
    raw_total = None

    if scope == 'explicit':
        raw = list(explicit or [])
        note = '手写清单'

    elif scope == 'all':
        raw = [it['symbol'] for it in
               _fetch_paged('/bds/symbol-infos', exclude_synced_today=_only_today)]
        note = '全市场（bds_symbol_info）'
        if _only_today:
            raw_total = _api_get('/bds/symbol-infos', limit=1, offset=0).get('total', 0)

    elif scope == 'industry':
        if not industry:
            raise ValueError('scope=industry 需填 SCOPE_INDUSTRY')
        raw = [it['symbol'] for it in
               _fetch_paged('/bds/symbol-infos', industry=industry,
                            exclude_synced_today=_only_today)]
        note = f'行业={industry}'
        if _only_today:
            raw_total = _api_get('/bds/symbol-infos', industry=industry,
                                 limit=1, offset=0).get('total', 0)

    elif scope == 'index':
        if not index_code:
            raise ValueError('scope=index 需填 SCOPE_INDEX_CODE')
        head = _api_get('/bds/index-constituents', index_code=index_code, limit=1, offset=0)
        items = head.get('items', [])
        if not items:
            raw, note = [], f'指数={index_code}（无成分数据）'
        else:
            td = items[0]['trade_date']
            rows = _fetch_paged('/bds/index-constituents', index_code=index_code, trade_date=td)
            raw = [it['symbol'] for it in rows]
            note = f'指数={index_code} @ {td}'
        if _only_today:
            # 成分股接口不返回 sync_at，改取「当日未同步集合」求交
            allowed = {it['symbol'] for it in
                       _fetch_paged('/bds/symbol-infos', exclude_synced_today=1)}
            raw_total = len(raw)
            raw = [s for s in raw if s in allowed]

    else:
        raise ValueError(f'未知 scope: {scope}')

    syms = _filter_market(raw)
    if max_symbols and max_symbols > 0:
        syms = syms[:max_symbols]

    tail = ''
    if exclude_synced_today:
        drop = (raw_total - len(raw)) if raw_total is not None else 0
        tail = f' | 当日已同步剔除 {max(drop, 0)} 只'
    print(f'symbol 来源: {note} | 原始 {len(raw)} -> 过滤后 {len(syms)}{tail}')
    return syms


def sync_one(symbol, table):
    '''调用单只同步接口，返回 {status, message}。

    status: success | no_data | error
    '''
    url = f'{API_BASE}/bds/sync/{table}'
    last_err = ''
    for attempt in range(1, MAX_RETRY + 1):
        try:
            r = SESSION.post(url, params={'symbol': symbol}, timeout=REQUEST_TIMEOUT)
            if r.status_code == 200:
                body = r.json()
                st = body.get('status')
                msg = body.get('message', '')
                if st in ('success', 'no_data'):
                    return {'status': st, 'message': msg}
                last_err = msg or 'sync error'  # status == error
            else:
                # HTTPException：body 为 {'detail': ...}
                try:
                    last_err = f'HTTP {r.status_code}: ' + str(r.json().get('detail', r.text[:200]))
                except Exception:
                    last_err = f'HTTP {r.status_code}: {r.text[:200]}'
        except Exception as e:
            last_err = f'{type(e).__name__}: {e}'
        if attempt < MAX_RETRY:
            time.sleep(RETRY_BACKOFF * attempt)
    return {'status': 'error', 'message': last_err}


def mark_synced(symbol):
    '''标记该 symbol 当日已完成全部目标表同步（写 bds_symbol_info.sync_at）。

    返回 {status, message}；status: ok | error
    '''
    url = f'{API_BASE}/bds/sync/mark-synced'
    try:
        r = SESSION.post(url, params={'symbol': symbol}, timeout=REQUEST_TIMEOUT)
        if r.status_code == 200:
            body = r.json()
            return {'status': body.get('status', 'error'),
                    'message': body.get('message', f"sync_at={body.get('sync_at')}")}
        return {'status': 'error', 'message': f'HTTP {r.status_code}: {r.text[:200]}'}
    except Exception as e:
        return {'status': 'error', 'message': f'{type(e).__name__}: {e}'}


print('工具函数就绪')


工具函数就绪


In [7]:
# ===================== 选择 symbol 来源 =====================
# scope 取值：
#   'explicit' 手写清单（SYMBOLS_EXPLICIT）
#   'all'      全市场（bds_symbol_info 全部股票）
#   'industry' 指定行业（SCOPE_INDUSTRY，行业名可用 /bds/symbol-industries 查看）
#   'index'    指数成分（SCOPE_INDEX_CODE，如 SHSE.000300）
SCOPE = 'index'

SYMBOLS_EXPLICIT = [
    # 'SHSE.600519',   # 贵州茅台
    # 'SZSE.000001',   # 平安银行
]
SCOPE_INDUSTRY = ''        # scope=industry 时填，如 '银行'
SCOPE_INDEX_CODE = 'SHSE.000300'      # scope=index 时填，如 'SHSE.000300'
MAX_SYMBOLS = 0            # >0 时只取前 N 只（试跑用，0=不限）

# True：剔除当日已完成同步的标的（依据 bds_symbol_info.sync_at，服务器本地当日）。
# 中途中断后直接重跑本 Cell 即可，当日已跑完的标的不会被重复请求。
EXCLUDE_SYNCED_TODAY = True

SYMBOLS = resolve_symbols(
    SCOPE,
    explicit=SYMBOLS_EXPLICIT,
    industry=SCOPE_INDUSTRY,
    index_code=SCOPE_INDEX_CODE,
    max_symbols=MAX_SYMBOLS,
    exclude_synced_today=EXCLUDE_SYNCED_TODAY,
)
print('预览:', SYMBOLS[:10], '...' if len(SYMBOLS) > 10 else '')
print('总任务数:', len(SYMBOLS) * len(TABLES), '（symbol 数 x 表数）')


symbol 来源: 指数=SHSE.000300 @ 2026-10-12 | 原始 283 -> 过滤后 283 | 当日已同步剔除 17 只
预览: ['SHSE.601377', 'SHSE.601901', 'SHSE.601688', 'SHSE.601788', 'SHSE.600809', 'SZSE.000768', 'SZSE.000776', 'SHSE.600893', 'SZSE.000792', 'SZSE.000807'] ...
总任务数: 1698 （symbol 数 x 表数）


In [ ]:
# ===================== 执行批量同步 =====================
# MARK_SYNCED_WHEN_DONE=True：一个 symbol 的全部表都非 error 后，写 sync_at=当日
MARK_SYNCED_WHEN_DONE = True


def run_tasks(tasks, mark_synced_when_done=MARK_SYNCED_WHEN_DONE):
    '''执行 (symbol, table) 任务列表，返回结果记录列表。

    按 symbol 分组推进：某个 symbol 的全部表都返回 success / no_data 时，
    调 /bds/sync/mark-synced 把 bds_symbol_info.sync_at 写为当日，
    下次运行即可用 EXCLUDE_SYNCED_TODAY=True 直接跳过。

    不做批次断点记录：中断后直接重跑整批即可，同步接口按增量起点
    MAX(date)+1 推进，已同步的表返回 no_data，幂等安全。
    '''
    global RESULTS
    RESULTS = []
    total = len(tasks)
    print(f'任务总数 {total}')
    if total == 0:
        print('无任务。')
        return RESULTS

    # 按 symbol 归并，保证同一标的的多张表连续执行（便于整只完成后打标记）
    by_symbol = {}
    for sym, table in tasks:
        by_symbol.setdefault(sym, []).append(table)

    t0 = time.time()
    fail_streak = 0
    i = 0
    marked = 0
    for sym, tables in by_symbol.items():
        sym_statuses = []
        for table in tables:
            i += 1
            res = sync_one(sym, table)
            RESULTS.append({'symbol': sym, 'table': table, **res})
            sym_statuses.append(res['status'])

            if res['status'] in ('success', 'no_data'):
                fail_streak = 0
            else:
                fail_streak += 1

            pause = random.uniform(SLEEP_MIN, SLEEP_MAX)
            if fail_streak >= FAIL_STREAK_LIMIT:
                pause += FAIL_STREAK_SLEEP
                print(f'  连续失败 {fail_streak} 次，额外停顿 {FAIL_STREAK_SLEEP:.0f}s')
                fail_streak = 0
            time.sleep(pause)

            if i % 10 == 0 or i == total:
                elapsed = time.time() - t0
                eta = elapsed / i * (total - i)
                print(f'[{i}/{total}] {sym} {table:15s} -> {res["status"]:7s} | '
                      f'已用 {elapsed:.0f}s 预计剩余 {eta:.0f}s | {res["message"][:40]}')

        # 该 symbol 全部表都非 error → 标记当日已同步
        if mark_synced_when_done and all(st in ('success', 'no_data') for st in sym_statuses):
            mk = mark_synced(sym)
            if mk['status'] == 'ok':
                marked += 1
            else:
                print(f'  [警告] {sym} 标记 sync_at 失败：{mk["message"][:60]}')

    print(f'完成，用时 {time.time() - t0:.0f}s | 已标记当日同步 {marked} 只')
    return RESULTS


TASKS = [(s, t) for s in SYMBOLS for t in TABLES]
RESULTS = run_tasks(TASKS)


任务总数 1698
[10/1698] SHSE.601901 finance-deriv   -> no_data | 已用 29s 预计剩余 4816s | 无数据可导入：SHSE.601901，请检查 gm 终端是否启动
[20/1698] SHSE.601788 fund-income     -> no_data | 已用 58s 预计剩余 4901s | 无数据可导入：SHSE.601788，请检查 gm 终端是否启动
[30/1698] SHSE.600809 daily-mktvalue  -> error   | 已用 83s 预计剩余 4642s | 同步失败：SHSE.600809，请检查 gm 终端是否启动
[40/1698] SZSE.000776 finance-deriv   -> no_data | 已用 107s 预计剩余 4429s | 无数据可导入：SZSE.000776，请检查 gm 终端是否启动
[50/1698] SZSE.000792 fund-income     -> no_data | 已用 135s 预计剩余 4455s | 无数据可导入：SZSE.000792，请检查 gm 终端是否启动
[60/1698] SZSE.000807 daily-mktvalue  -> error   | 已用 165s 预计剩余 4510s | 同步失败：SZSE.000807，请检查 gm 终端是否启动
[70/1698] SHSE.601398 finance-deriv   -> no_data | 已用 191s 预计剩余 4445s | 无数据可导入：SHSE.601398，请检查 gm 终端是否启动
[80/1698] SHSE.600426 fund-income     -> success | 已用 260s 预计剩余 5255s | 同步完成：SHSE.600426，更新 102 条
[90/1698] SHSE.600031 daily-mktvalue  -> success | 已用 362s 预计剩余 6461s | 同步完成：SHSE.600031，更新 5654 条
[100/1698] SHSE.600588 finance-deriv   -> success | 已用 427s 预计剩余 

In [ ]:
# ===================== 结果汇总 =====================
df = pd.DataFrame(RESULTS)
print('本次执行条数:', len(df))
if not df.empty:
    print('\n状态分布：')
    print(df['status'].value_counts().to_string())

    print('\n各表 x 状态：')
    pivot = (df.pivot_table(index='table', columns='status', values='symbol', aggfunc='count')
               .fillna(0).astype(int))
    print(pivot.to_string())

    err = df[df['status'] == 'error']
    if not err.empty:
        print(f'\n失败明细（前 20，共 {len(err)}）：')
        print(err.head(20).to_string())

    df.to_csv(RESULT_CSV, index=False, encoding='utf-8-sig')
    print(f'\n结果已落盘: {RESULT_CSV}')


In [ ]:
# ===================== 仅重试失败项 =====================
# 重跑上一次执行中 status=error 的 (symbol, table)
RETRY_NO_DATA = True   # True: 连同 no_data（可能是 gm 未启动）一并重试

res_df = pd.DataFrame(RESULTS)
if RETRY_NO_DATA:
    failed = res_df[res_df['status'].isin(['error', 'no_data'])]
else:
    failed = res_df[res_df['status'] == 'error']
retry_tasks = list(dict.fromkeys(zip(failed['symbol'], failed['table'])))
print('待重试任务数:', len(retry_tasks))

if retry_tasks:
    df2 = pd.DataFrame(run_tasks(retry_tasks))
    RESULTS = RESULTS + df2.to_dict('records')
    print('重试后状态分布：')
    print(pd.DataFrame(RESULTS)['status'].value_counts().to_string())
else:
    print('无失败项。')
